# ASG Airlines — End-to-End Data Engineering Pipeline

**Objective:** Ingest raw operational flight, booking, payment, and passenger data; clean and standardize it;
mask PII; transform it into an analytics-ready model; and compute the operational KPIs required for the
Power BI dashboard.

**Source file:** `data/raw/UseCase_-_Airlines.xlsx` — 4 sheets: `flights`, `bookings`, `payments`, `passengers`.

**Pipeline stages**
1. Ingestion & data-quality profiling
2. Cleaning & standardization — `flights`
3. Cleaning & standardization — `bookings`
4. Cleaning & standardization — `payments`
5. Cleaning & standardization — `passengers`
6. PII masking
7. KPI computation
8. Export — cleaned dataset (Power BI-ready) + anomaly log (data governance)

> **Scope note:** the problem statement describes a single flights table, but the provided workbook
> includes three additional relational tables (`bookings`, `payments`, `passengers`) carrying passenger
> PII (passport number, Aadhaar ID, contact details). This pipeline treats `flights` as the core fact
> table for the required operational KPIs, and processes the other three tables to satisfy the PII-masking
> and privacy-and-access-control requirements, plus a small set of bonus commercial KPIs.


In [1]:
import pandas as pd
import numpy as np
import hashlib
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("asg_airlines_pipeline")

RAW = Path("../data/raw/UseCase_-_Airlines.xlsx")
OUT = Path("../output/cleaned")
LOG = Path("../output/anomaly_logs")
OUT.mkdir(parents=True, exist_ok=True)
LOG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Ingestion & Data-Quality Profiling

Read all four sheets and run an initial profile pass. This is the "data quality checks during ingestion"
step called for in the brief — we log null counts, duplicate keys, and referential integrity between
tables *before* touching the data, so the cleaning decisions below are evidence-based rather than assumed.


In [2]:
xls = pd.ExcelFile(RAW)
flights = pd.read_excel(xls, "flights")
bookings = pd.read_excel(xls, "bookings")
payments = pd.read_excel(xls, "payments")
passengers = pd.read_excel(xls, "passengers")

logger.info(f"Loaded flights={flights.shape}, bookings={bookings.shape}, "
            f"payments={payments.shape}, passengers={passengers.shape}")

print("Null counts:")
for name, df in [("flights", flights), ("bookings", bookings),
                  ("payments", payments), ("passengers", passengers)]:
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"\n{name}:")
    print(nulls if len(nulls) else "  (none)")


2026-09-06 06:43:47,690 | INFO | Loaded flights=(1020, 7), bookings=(1000, 9), payments=(1000, 4), passengers=(1039, 9)


Null counts:

flights:
airline    41
dtype: int64

bookings:
status    45
dtype: int64

payments:
amount    48
dtype: int64

passengers:
last_name    10
dtype: int64


In [3]:
# Referential integrity check
orphan_bookings_flight = (~bookings.flight_id.isin(flights.flight_id)).sum()
orphan_bookings_pax = (~bookings.passenger_id.isin(passengers.passenger_id)).sum()
orphan_payments = (~payments.booking_id.isin(bookings.booking_id)).sum()

print(f"Bookings referencing a missing flight_id:    {orphan_bookings_flight}")
print(f"Bookings referencing a missing passenger_id: {orphan_bookings_pax}")
print(f"Payments referencing a missing booking_id:   {orphan_payments}")
print()
print(f"Duplicate flight_id values:    {flights.flight_id.duplicated().sum()} of {len(flights)} rows")
print(f"Duplicate passenger_id values: {passengers.passenger_id.duplicated().sum()} of {len(passengers)} rows")
print(f"Duplicate booking_id values:   {bookings.booking_id.duplicated().sum()} of {len(bookings)} rows")
print(f"Duplicate payment_id values:   {payments.payment_id.duplicated().sum()} of {len(payments)} rows")


Bookings referencing a missing flight_id:    0
Bookings referencing a missing passenger_id: 0
Payments referencing a missing booking_id:   0

Duplicate flight_id values:    16 of 1020 rows
Duplicate passenger_id values: 39 of 1039 rows
Duplicate booking_id values:   0 of 1000 rows
Duplicate payment_id values:   0 of 1000 rows


In [4]:
# Central anomaly log — every cleaning decision below writes here.
# This becomes the data-governance audit trail and doubles as the "Delays / Anomalies" KPI input,
# since the dataset has no scheduled-vs-actual time field to compute a true delay from (see Section 6).
anomaly_records = []

def log_anomaly(table, key, issue, detail=""):
    anomaly_records.append({"table": table, "record_key": key, "issue": issue, "detail": detail})


## 2. Cleaning & Standardization — `flights`

Issues found during profiling and how each is handled:

| Issue | Found | Handling |
|---|---|---|
| Exact duplicate rows (same flight_id + all fields) | 15 pairs | Drop, keep first occurrence |
| `flight_id` reused across two *different* flights | 1 id (2 rows) | Disambiguate with a `-DUPn` suffix; identifier collision is a genuine data-corruption case, not a duplicate to drop |
| Missing / literal `"UNKNOWN"` airline | 69 rows | Normalized to a single explicit `UNKNOWN` category (not imputed — multiple airlines fly every route in this data, so a route-based guess would be fabricated, not inferred) |
| Overnight (cross-day) flights | 122 rows | `departure_time`/`arrival_time` are full timestamps, so `arrival − departure` already yields the correct positive duration across midnight — no special-casing needed once dates are trusted |
| Corrupted arrival date (arrival timestamp *before* departure) | 1 row | Time-of-day is kept, but the arrival *date* is rolled forward a day at a time until duration falls in the plausible 0–8h domestic range; the row is flagged, not silently fixed |
| Unreliable source `duration` column (mixed Excel time/datetime types, wrong on the corrupted row) | — | Dropped; `duration_hours` is recomputed directly from the corrected timestamps |


In [5]:
f = flights.copy()

# 2a. Exact full-row duplicates -> drop, log
exact_dupe_mask = f.duplicated(keep="first")
for idx in f[exact_dupe_mask].index:
    log_anomaly("flights", f.loc[idx, "flight_id"], "exact_duplicate_row_dropped")
f = f[~exact_dupe_mask].copy()
logger.info(f"Dropped {exact_dupe_mask.sum()} exact-duplicate flight rows -> {len(f)} remain")


2026-09-06 06:43:47,719 | INFO | Dropped 15 exact-duplicate flight rows -> 1005 remain


In [6]:
# 2b. Remaining flight_id collisions (same id, genuinely different flight) -> disambiguate
dupe_ids = f[f.duplicated("flight_id", keep=False)]["flight_id"].unique()
for fid in dupe_ids:
    rows = f[f["flight_id"] == fid]
    for n, idx in enumerate(rows.index, start=1):
        new_id = f"{fid}-DUP{n}"
        f.loc[idx, "flight_id"] = new_id
        log_anomaly("flights", fid, "flight_id_collision_disambiguated", f"reassigned -> {new_id}")

assert f["flight_id"].duplicated().sum() == 0, "flight_id must be unique after disambiguation"
logger.info(f"Disambiguated {len(dupe_ids)} colliding flight_id group(s)")


2026-09-06 06:43:47,726 | INFO | Disambiguated 1 colliding flight_id group(s)


In [7]:
# 2c. Missing / unknown airline -> normalize to one explicit category
f["airline_reported_missing"] = f["airline"].isna() | (f["airline"] == "UNKNOWN")
for idx in f[f["airline_reported_missing"]].index:
    log_anomaly("flights", f.loc[idx, "flight_id"], "airline_missing_or_unknown")
f["airline"] = f["airline"].fillna("UNKNOWN")
logger.info(f"Flagged {f['airline_reported_missing'].sum()} flights with missing/unknown airline")


2026-09-06 06:43:47,735 | INFO | Flagged 69 flights with missing/unknown airline


In [8]:
# 2d. Recompute duration from timestamps; correct corrupted (reversed) arrival dates
MAX_PLAUSIBLE_HOURS = 8  # domestic sector ceiling observed in the clean data (~5h max) + safety buffer

def fix_duration(row):
    dep, arr = row["departure_time"], row["arrival_time"]
    dur = (arr - dep).total_seconds() / 3600
    if 0 <= dur <= MAX_PLAUSIBLE_HOURS:
        return dur, arr, False
    # corrupted: walk the arrival date forward until duration is plausible again
    corrected_arr = arr
    for _ in range(3):
        corrected_arr = corrected_arr + pd.Timedelta(days=1)
        dur = (corrected_arr - dep).total_seconds() / 3600
        if 0 <= dur <= MAX_PLAUSIBLE_HOURS:
            return dur, corrected_arr, True
    return dur, corrected_arr, True  # fallback: still flagged for manual review

results = f.apply(fix_duration, axis=1, result_type="expand")
results.columns = ["duration_hours", "arrival_time_corrected", "arrival_date_corrected_flag"]
f = pd.concat([f, results], axis=1)
f["arrival_time"] = f["arrival_time_corrected"]
f["is_overnight"] = f["arrival_time"].dt.date > f["departure_time"].dt.date
f = f.drop(columns=["arrival_time_corrected", "duration"])

for idx in f[f["arrival_date_corrected_flag"]].index:
    log_anomaly("flights", f.loc[idx, "flight_id"], "corrupted_arrival_date_corrected",
                f"duration_hours={f.loc[idx,'duration_hours']:.2f}")

logger.info(f"Corrected {f['arrival_date_corrected_flag'].sum()} corrupted arrival date(s)")
assert (f["duration_hours"] < 0).sum() == 0, "no negative durations should remain"
f[["flight_id","airline","source","destination","departure_time","arrival_time","duration_hours","is_overnight"]].head()


2026-09-06 06:43:47,765 | INFO | Corrected 1 corrupted arrival date(s)


,flight_id,airline,source,destination,departure_time,arrival_time,duration_hours,is_overnight
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,2.900000,True
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,1.800000,True
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,1.750000,True
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,2.600000,True
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,4.983333,True


In [9]:
# 2e. Route-level statistical anomaly flag — the closest available proxy for "delay",
# since the source data has no scheduled-vs-actual time field (see Section 6 note).
f["route"] = f["source"] + "-" + f["destination"]
route_stats = f.groupby("route")["duration_hours"].agg(["mean", "std"]).rename(
    columns={"mean": "route_mean_hours", "std": "route_std_hours"})
f = f.merge(route_stats, on="route", how="left")
f["route_std_hours"] = f["route_std_hours"].fillna(0)
f["duration_anomaly_flag"] = (
    (f["duration_hours"] - f["route_mean_hours"]).abs() > 2 * f["route_std_hours"].replace(0, np.inf)
)
logger.info(f"Flagged {f['duration_anomaly_flag'].sum()} statistical duration outlier(s) by route")

flights_clean = f.drop(columns=["arrival_date_corrected_flag"]).copy()
flights_clean.describe(include="all").T


2026-09-06 06:43:47,788 | INFO | Flagged 1 statistical duration outlier(s) by route


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
flight_id,1005,1005,SJ010,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
airline,1005,5,IndiGo,249,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,1005,6,BOM,205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
destination,1005,6,DEL,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
departure_time,1005,NaN,NaN,NaN,2026-04-19 06:49:14.539209,2026-04-17 12:25:41.701000,2026-04-18 10:43:41.701000,2026-04-19 07:07:41.703000,2026-04-20 03:22:41.704000,2026-04-20 23:38:41.701000,NaN
arrival_time,1005,NaN,NaN,NaN,2026-04-19 09:33:51.741245,2026-04-17 14:10:41.703000,2026-04-18 13:14:41.701000,2026-04-19 09:40:41.702000,2026-04-20 06:08:41.702000,2026-04-21 04:04:41.703000,NaN
airline_reported_missing,1005,2,False,936,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration_hours,1005.0,NaN,NaN,NaN,2.743667,0.5,1.65,2.766667,3.9,5.0,1.291064
is_overnight,1005,2,False,883,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route,1005,30,BOM-CCU,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Cleaning & Standardization — `bookings`

`status` contains both true nulls and the literal string `"INVALID"` — both are the same underlying
problem (status not captured/parseable at source), so both are normalized to one `UNKNOWN` category
rather than dropping the row (the booking's flight/passenger/payment links are still valid and useful).


In [10]:
b = bookings.copy()
b["status_reported_missing"] = b["status"].isna() | (b["status"] == "INVALID")
for idx in b[b["status_reported_missing"]].index:
    log_anomaly("bookings", b.loc[idx, "booking_id"], "status_missing_or_invalid")
b["status"] = b["status"].fillna("UNKNOWN").replace("INVALID", "UNKNOWN")
logger.info(f"Normalized {b['status_reported_missing'].sum()} missing/invalid booking statuses")
b["status"].value_counts()


2026-09-06 06:43:47,815 | INFO | Normalized 75 missing/invalid booking statuses


status
CONFIRMED    320
CANCELLED    314
PENDING      291
UNKNOWN       75
Name: count, dtype: int64

## 4. Cleaning & Standardization — `payments`

`amount` mixes genuine nulls with the literal string `"INVALID"`. Both are coerced to `NaN` and flagged
— the transaction row is kept (payment method and booking linkage are still valid) but the value is
excluded from any revenue aggregation.


In [11]:
p = payments.copy()
p["amount_reported_invalid"] = ~pd.to_numeric(p["amount"], errors="coerce").notna()
for idx in p[p["amount_reported_invalid"]].index:
    log_anomaly("payments", p.loc[idx, "payment_id"], "amount_missing_or_invalid",
                f"raw_value={p.loc[idx,'amount']!r}")
p["amount"] = pd.to_numeric(p["amount"], errors="coerce")
logger.info(f"Flagged {p['amount_reported_invalid'].sum()} missing/invalid payment amounts")
p["amount"].describe()


2026-09-06 06:43:47,826 | INFO | Flagged 78 missing/invalid payment amounts


count      922.000000
mean      8009.916464
std       4022.055392
min       1002.590000
25%       4623.495000
50%       8027.125000
75%      11288.155000
max      14992.950000
Name: amount, dtype: float64

## 5. Cleaning & Standardization — `passengers`

`passenger_id` — which should be a unique natural key — is reused across genuinely different individuals
(different email, sometimes different name) for 39 rows. Since the analytics dimension needs a unique
key to join against safely, the **first** occurrence of each `passenger_id` is kept as the canonical
dimension row; the conflicting duplicates are not silently discarded — they're written to a separate
`passenger_id_conflicts.csv` for governance review, since renaming/merging them without more context
would be a guess.

`aadhaar_id` values are also inconsistent in length (a valid Aadhaar number is always 12 digits; this
data has some 10- and 11-digit values) — flagged as a format anomaly.


In [12]:
ps = passengers.copy()
dupe_pid_mask = ps.duplicated("passenger_id", keep="first")
for idx in ps[dupe_pid_mask].index:
    log_anomaly("passengers", ps.loc[idx, "passenger_id"], "passenger_id_reused_conflicting_record",
                f"name={ps.loc[idx,'first_name']} {ps.loc[idx,'last_name']}, email={ps.loc[idx,'email']}")

passenger_id_conflicts = ps[dupe_pid_mask].copy()      # kept for governance review, not used downstream
passengers_clean = ps[~dupe_pid_mask].copy()            # 1 row per passenger_id -> safe BI dimension key
passengers_clean["last_name"] = passengers_clean["last_name"].fillna("UNKNOWN")

passengers_clean["aadhaar_invalid_format"] = passengers_clean["aadhaar_id"].astype(str).str.len() != 12
for idx in passengers_clean[passengers_clean["aadhaar_invalid_format"]].index:
    log_anomaly("passengers", passengers_clean.loc[idx, "passenger_id"], "aadhaar_id_invalid_length",
                f"len={len(str(passengers_clean.loc[idx,'aadhaar_id']))}")

logger.info(f"Passengers: {dupe_pid_mask.sum()} conflicting duplicate rows set aside, "
            f"{passengers_clean['aadhaar_invalid_format'].sum()} invalid-length Aadhaar IDs flagged")


2026-09-06 06:43:47,845 | INFO | Passengers: 39 conflicting duplicate rows set aside, 109 invalid-length Aadhaar IDs flagged


## 6. PII Masking

The workbook carries several fields that are Personally Identifiable Information under most privacy
frameworks: **Aadhaar ID**, **passport number**, **email**, **phone**, and **emergency contact
name/phone**. Masking strategy, by field:

| Field | Technique | Why |
|---|---|---|
| `aadhaar_id`, `passport_number` | Salted SHA-256, truncated to 16 hex chars | Irreversible one-way hash — preserves uniqueness for joins/dedup checks without ever exposing or being able to recover the real number |
| `email` | Partial masking (`j***n@domain.com`) | Keeps the value human-recognizable for support/debugging without exposing the full address |
| `phone` | Partial masking (first 6 / last 2 digits shown) | Same rationale as email |
| `emergency_contact_name`, `emergency_contact_phone` | **Dropped entirely** | Data-minimization: neither field is required by any KPI or dashboard visual, so the lowest-risk action is to not carry them into the analytics layer at all, rather than mask data that serves no analytical purpose |

For this submission, masking runs locally in Python with no cloud dependency — the pipeline stays fully
reproducible without an Azure subscription. The hashing salt (`ASG_MASK_SALT`) is read from an
**environment variable**, not hardcoded: the cell below raises an error rather than silently falling
back to a default if it isn't set, so a real secret can never end up committed to source control by
accident. In a production Azure deployment, that same environment variable would be populated from
**Azure Key Vault** at pipeline start-up rather than set by hand — Key Vault is the production design
target, not something this submission depends on to run.


In [13]:
import os

ASG_MASK_SALT = os.getenv("ASG_MASK_SALT")
if not ASG_MASK_SALT:
    raise ValueError(
        "ASG_MASK_SALT is not set. Set it as an environment variable before running this "
        "notebook, e.g.:  export ASG_MASK_SALT='your-local-secret'   (see .env.example)\n"
        "In production this value comes from Azure Key Vault at runtime, never hardcoded here."
    )

def sha256_mask(value, salt=ASG_MASK_SALT):
    if pd.isna(value):
        return None
    return hashlib.sha256(f"{salt}{value}".encode()).hexdigest()[:16]

def partial_mask_email(email):
    if pd.isna(email):
        return None
    try:
        local, domain = email.split("@")
    except ValueError:
        return "***"
    masked_local = local[0] + "*" if len(local) <= 2 else local[0] + "*" * (len(local) - 2) + local[-1]
    return f"{masked_local}@{domain}"

def partial_mask_phone(phone):
    if pd.isna(phone):
        return None
    phone = str(phone)
    return phone[:6] + "*" * (len(phone) - 8) + phone[-2:] if len(phone) > 8 else "*" * len(phone)


In [14]:
bookings_masked = b.drop(columns=["passport_number", "emergency_contact_name", "emergency_contact_phone"]).copy()
bookings_masked["passport_number_hash"] = b["passport_number"].apply(sha256_mask)

passengers_masked = passengers_clean.drop(columns=["aadhaar_id", "email", "phone"]).copy()
passengers_masked["aadhaar_id_hash"] = passengers_clean["aadhaar_id"].apply(sha256_mask)
passengers_masked["email_masked"] = passengers_clean["email"].apply(partial_mask_email)
passengers_masked["phone_masked"] = passengers_clean["phone"].apply(partial_mask_phone)

logger.info("PII masking complete: passport & Aadhaar hashed, email & phone partially masked, "
            "emergency contact fields dropped")
passengers_masked[["passenger_id","first_name","last_name","email_masked","phone_masked","aadhaar_id_hash"]].head()


2026-09-06 06:43:47,864 | INFO | PII masking complete: passport & Aadhaar hashed, email & phone partially masked, emergency contact fields dropped


,passenger_id,first_name,last_name,email_masked,phone_masked,aadhaar_id_hash
0,P1000,Vivaan,Chatterjee,v***************e@gmail.com,+91-68******90,dbc976463c58cebc
1,P1001,Krishna,Reddy,k***********y@hotmail.com,+91-67******97,770d0e3801d16861
2,P1002,Myra,Naidu,m********u@outlook.com,+91-61******92,d94d5c74b1b87dfe
3,P1003,Myra,Mishra,m*********a@hotmail.com,+91-87******51,c0365e092e3fd791
4,P1004,Saanvi,Banerjee,s*************e@outlook.com,+91-78******13,e15e595420702d9c


## 7. KPI Computation

Covers the four required KPIs (Average Flight Duration, Route-wise Traffic, Delays/Anomalies,
Distribution of Flights by Airline) plus a few additional commercial KPIs enabled by the
bookings/payments tables.

**Note on "Delays / Anomalies":** the dataset contains only *actual* departure/arrival timestamps —
there is no scheduled-time field to diff against, so a true on-time-performance delay cannot be
computed from this data. The KPI is therefore built from two things that *are* measurable: (1) the
full **data-quality anomaly log** built during cleaning (corrupted IDs, corrupted dates, missing
fields), and (2) a **statistical duration outlier flag** per route (>2σ from that route's mean
duration) as an operational-anomaly proxy. Both are documented as an explicit assumption in the report.


In [15]:
fc = flights_clean

kpi_overall_duration = pd.DataFrame({
    "metric": ["Average Flight Duration (hrs)", "Median Flight Duration (hrs)",
               "Min Duration (hrs)", "Max Duration (hrs)", "Overnight Flights", "Total Flights"],
    "value": [fc["duration_hours"].mean(), fc["duration_hours"].median(),
              fc["duration_hours"].min(), fc["duration_hours"].max(),
              int(fc["is_overnight"].sum()), len(fc)]
})
kpi_overall_duration


,metric,value
0,Average Flight Duration (hrs),2.743667
1,Median Flight Duration (hrs),2.766667
2,Min Duration (hrs),0.500000
3,Max Duration (hrs),5.000000
4,Overnight Flights,122.000000
5,Total Flights,1005.000000


In [16]:
kpi_duration_by_airline = (fc.groupby("airline")["duration_hours"]
    .agg(avg_duration_hours="mean", flight_count="count")
    .reset_index().sort_values("flight_count", ascending=False))
kpi_duration_by_airline


,airline,avg_duration_hours,flight_count
1,IndiGo,2.795049,249
2,SpiceJet,2.736513,236
0,Air India,2.719101,233
4,Vistara,2.711318,218
3,UNKNOWN,2.767874,69


In [17]:
kpi_route_traffic = (fc.groupby(["source", "destination", "route"])
    .agg(flight_count=("flight_id", "count"),
         avg_duration_hours=("duration_hours", "mean"),
         anomaly_count=("duration_anomaly_flag", "sum"))
    .reset_index().sort_values("flight_count", ascending=False))
kpi_route_traffic.head(10)


,source,destination,route,flight_count,avg_duration_hours,anomaly_count
6,BOM,CCU,BOM-CCU,90,2.825187,0
12,CCU,DEL,CCU-DEL,72,2.560186,0
25,MAA,BLR,MAA-BLR,65,2.880259,0
0,BLR,BOM,BLR-BOM,60,2.462226,0
24,HYD,MAA,HYD-MAA,57,2.546788,0
18,DEL,HYD,DEL-HYD,54,2.912654,0
23,HYD,DEL,HYD-DEL,42,3.089290,0
7,BOM,DEL,BOM-DEL,39,2.563677,0
11,CCU,BOM,CCU-BOM,33,2.732833,0
15,DEL,BLR,DEL-BLR,29,2.833914,0


In [18]:
kpi_airline_distribution = fc["airline"].value_counts().reset_index()
kpi_airline_distribution.columns = ["airline", "flight_count"]
kpi_airline_distribution["pct_of_total"] = (kpi_airline_distribution["flight_count"] / len(fc) * 100).round(2)
kpi_airline_distribution


,airline,flight_count,pct_of_total
0,IndiGo,249,24.78
1,SpiceJet,236,23.48
2,Air India,233,23.18
3,Vistara,218,21.69
4,UNKNOWN,69,6.87


In [19]:
anomaly_df = pd.DataFrame(anomaly_records)
kpi_anomalies_summary = (anomaly_df.groupby("issue").size()
    .reset_index(name="count").sort_values("count", ascending=False))
kpi_anomalies_summary


,issue,count
0,aadhaar_id_invalid_length,109
2,amount_missing_or_invalid,78
7,status_missing_or_invalid,75
1,airline_missing_or_unknown,69
6,passenger_id_reused_conflicting_record,39
4,exact_duplicate_row_dropped,15
5,flight_id_collision_disambiguated,2
3,corrupted_arrival_date_corrected,1


In [20]:
# Bonus KPIs — booking status mix, revenue by payment method, per-flight booking (load) count
kpi_booking_status = bookings_masked["status"].value_counts().reset_index()
kpi_booking_status.columns = ["status", "booking_count"]

kpi_revenue_by_method = p.groupby("payment_method")["amount"].agg(
    total_revenue="sum", avg_amount="mean", tx_count="count").reset_index()

bookings_per_flight = bookings_masked.groupby("flight_id").size().reset_index(name="booking_count")
kpi_load_by_flight = fc.merge(bookings_per_flight, on="flight_id", how="left")
kpi_load_by_flight["booking_count"] = kpi_load_by_flight["booking_count"].fillna(0)

print(kpi_booking_status, "\n")
print(kpi_revenue_by_method)


      status  booking_count
0  CONFIRMED            320
1  CANCELLED            314
2    PENDING            291
3    UNKNOWN             75 

  payment_method  total_revenue   avg_amount  tx_count
0           CARD     2400812.04  8002.706800       300
1     NETBANKING     2365644.17  8129.361409       291
2            UPI     2618686.77  7911.440393       331


## 8. Export — Cleaned Dataset & Anomaly Log

Two workbooks are written to `output/`:

- **`cleaned_dataset.xlsx`** — the analytics-ready, PII-masked tables (`fact_flights`, `dim_bookings`,
  `fact_payments`, `dim_passengers`). This is what Power BI imports.
- **`kpi_summary.xlsx`** — pre-aggregated KPI tables, useful as a sanity check against the Power BI
  measures and as a fallback if the dashboard needs static figures.

The anomaly log and the passenger-ID conflict list are written separately to `output/anomaly_logs/` as
CSVs — this is the data-governance audit trail referenced in the documentation.


In [21]:
with pd.ExcelWriter(OUT / "cleaned_dataset.xlsx", engine="openpyxl") as writer:
    flights_clean.to_excel(writer, sheet_name="fact_flights", index=False)
    bookings_masked.to_excel(writer, sheet_name="dim_bookings", index=False)
    p.to_excel(writer, sheet_name="fact_payments", index=False)
    passengers_masked.to_excel(writer, sheet_name="dim_passengers", index=False)

with pd.ExcelWriter(OUT / "kpi_summary.xlsx", engine="openpyxl") as writer:
    kpi_overall_duration.to_excel(writer, sheet_name="overall_duration", index=False)
    kpi_duration_by_airline.to_excel(writer, sheet_name="duration_by_airline", index=False)
    kpi_route_traffic.to_excel(writer, sheet_name="route_traffic", index=False)
    kpi_airline_distribution.to_excel(writer, sheet_name="airline_distribution", index=False)
    kpi_anomalies_summary.to_excel(writer, sheet_name="anomalies_summary", index=False)
    kpi_booking_status.to_excel(writer, sheet_name="booking_status", index=False)
    kpi_revenue_by_method.to_excel(writer, sheet_name="revenue_by_payment_method", index=False)

anomaly_df.to_csv(LOG / "anomaly_log.csv", index=False)
passenger_id_conflicts.to_csv(LOG / "passenger_id_conflicts.csv", index=False)

logger.info(f"Wrote cleaned_dataset.xlsx and kpi_summary.xlsx to {OUT.resolve()}")
logger.info(f"Wrote anomaly_log.csv ({len(anomaly_df)} rows) and passenger_id_conflicts.csv "
            f"({len(passenger_id_conflicts)} rows) to {LOG.resolve()}")


2026-09-06 06:43:48,430 | INFO | Wrote cleaned_dataset.xlsx and kpi_summary.xlsx to /home/claude/asg_airlines/output/cleaned


2026-09-06 06:43:48,431 | INFO | Wrote anomaly_log.csv (388 rows) and passenger_id_conflicts.csv (39 rows) to /home/claude/asg_airlines/output/anomaly_logs


## 9. Pipeline Summary

Quick end-to-end sanity check before handing off to Power BI.


In [22]:
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"flights:    {len(flights):>5} raw -> {len(flights_clean):>5} clean "
      f"({len(flights) - len(flights_clean)} exact duplicates removed)")
print(f"bookings:   {len(bookings):>5} raw -> {len(bookings_masked):>5} clean")
print(f"payments:   {len(payments):>5} raw -> {len(p):>5} clean")
print(f"passengers: {len(passengers):>5} raw -> {len(passengers_masked):>5} clean "
      f"({dupe_pid_mask.sum()} conflicting duplicate IDs set aside)")
print()
print(f"Total anomalies logged: {len(anomaly_df)}")
print(anomaly_df['issue'].value_counts().to_string())
print()
print(f"Avg flight duration: {fc['duration_hours'].mean():.2f} hrs")
print(f"Statistical duration outliers flagged: {fc['duration_anomaly_flag'].sum()}")


PIPELINE SUMMARY
flights:     1020 raw ->  1005 clean (15 exact duplicates removed)
bookings:    1000 raw ->  1000 clean
payments:    1000 raw ->  1000 clean
passengers:  1039 raw ->  1000 clean (39 conflicting duplicate IDs set aside)

Total anomalies logged: 388
issue
aadhaar_id_invalid_length                 109
amount_missing_or_invalid                  78
status_missing_or_invalid                  75
airline_missing_or_unknown                 69
passenger_id_reused_conflicting_record     39
exact_duplicate_row_dropped                15
flight_id_collision_disambiguated           2
corrupted_arrival_date_corrected            1

Avg flight duration: 2.74 hrs
Statistical duration outliers flagged: 1
